# B10 · Counting points and density maps: where are the landslides?

> **Time:** about 2 hours  
> **You need:** Part A and Lessons B1 to B9, especially spatial joins and counting points per polygon (B9) and UTM 45N (B4, B8).  
> **By the end you can:**
> 1. turn points into counts and **densities** per 100 km², per district and per grid cell,
> 2. build square and hexagon grids in UTM 45N and count points in every cell,
> 3. classify a choropleth with **mapclassify** (equal interval, quantiles, natural breaks) and write an honest legend.

A map of dots shows where things are, but it is poor at answering "where are there **most**?". This lesson turns the practice landslide points into counts and densities, on districts and on regular grids. Then it shows how the choice of map classes can make the same numbers tell different stories. The landslide records are made-up practice data, so the densities describe the practice file, not Nepal.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 20 of 21 · [Course home](../README.md)  
← Previous: [B9 · Spatial joins and overlays: landslides, districts and headquarters](B09_Spatial_joins_and_overlays.ipynb) · Next: [B11 · Making a beautiful map: a finished landslide map of Nepal](B11_Making_a_beautiful_map.ipynb) →

## 1. Why count points in areas?

Dots on a map have two weaknesses. Close points hide each other, so 9 landslides can look like 3. And the eye cannot compare places fairly.

**Aggregating** fixes both: you count the points in each area and colour the areas by their count. The areas can be districts (step 2) or a regular grid (steps 3 to 5).

The counting recipe is the one from B9: `sjoin` → `groupby(...).size()` → `merge(how="left")` → `fillna(0)`. Below, everything is projected to UTM 45N straight away, because steps 2 to 5 measure areas.

As in B6, fix the Chitwan spelling first.

In [ ]:
import math
import numpy as np
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import box, Polygon

DATA = Path("..") / "data"
districts_utm = gpd.read_file(DATA / "nepal_districts.geojson").to_crs(32645)

slides = pd.read_csv(DATA / "practice_landslides.csv")
slides["district"] = slides["district"].replace("Chitwan", "Chitawan")   # as in B6
landslides_utm = gpd.GeoDataFrame(
    slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]), crs="EPSG:4326"
).to_crs(32645)

joined = gpd.sjoin(landslides_utm, districts_utm, how="inner", predicate="within")
counts = joined.groupby("district_code").size().reset_index(name="n_landslides")
district_counts = districts_utm.merge(counts, on="district_code", how="left")
district_counts["n_landslides"] = district_counts["n_landslides"].fillna(0).astype(int)
print(district_counts["n_landslides"].sum(), len(district_counts))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
districts_utm.plot(ax=ax1, color="whitesmoke", edgecolor="grey", linewidth=0.3)
landslides_utm.plot(ax=ax1, color="crimson", markersize=8)
ax1.set_title("Dots: 40 points, some hidden")
district_counts.plot(ax=ax2, column="n_landslides", cmap="Reds", legend=True,
                     edgecolor="grey", linewidth=0.3, legend_kwds={"shrink": 0.6})
ax2.set_title("Counts per district")
ax1.set_axis_off()
ax2.set_axis_off()
plt.show()

All 40 points are counted, and all 77 districts are kept. On the dot map, the 9 Dolakha points almost merge into one smudge; on the count map, Dolakha and Chitawan stand out at once.

The maps are drawn in UTM 45N, so Nepal looks slightly tilted compared with the degree maps of B1 to B3. Both are correct; they are different projections.

### Practice

**Q1 · one line:** How many districts have at least 5 landslide records?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(district_counts["n_landslides"] >= 5).sum()
```

4 districts: Chitawan and Dolakha with 9, Myagdi and Rasuwa with 5.
</details>

**Q2 · two lines:** Count only the **rainfall** landslides per district: filter `joined` by `trigger`, then group by `district_right` and count with `.size()`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
rain_slides = joined[joined["trigger"] == "rainfall"]
print(rain_slides.groupby("district_right").size())
```

All 8 districts appear, but the numbers change: Chitawan and Dolakha have 8 rainfall records each, while Taplejung, whose records are mostly road cuts, has only 1.
</details>

**Q3 · three lines:** Make a dot map of the landslides coloured by trigger: plot `districts_utm` in light grey, then `landslides_utm` with `column="trigger"` and `legend=True`. As in A10, lines that only set up the plot (`fig, ax = plt.subplots()` and `plt.show()`) do not count toward the line count.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(9, 5))
districts_utm.plot(ax=ax, color="whitesmoke", edgecolor="grey", linewidth=0.3)
landslides_utm.plot(ax=ax, column="trigger", legend=True, markersize=15)
ax.set_axis_off()
plt.show()
```

`column=` works for text columns too: each trigger gets its own colour and a legend entry. Even with colours, overlapping dots still hide each other, which is why the rest of this lesson counts them.
</details>

## 2. Density: counts per 100 km²

Districts differ a lot in size. Nine landslides in a small district mean more than nine in a large one.

**Density** divides the count by the area: `n / area_km2`. The result is tiny here, so we multiply by 100 and report **landslides per 100 km²**. Always state the unit.

Measure the area in a projected CRS, here UTM 45N, never in degrees (B4). `.area` gives m²; divide by 1,000,000 for km².

**Predict:** Chitawan and Dolakha both have 9 records. Which one has the higher density?

In [ ]:
district_counts["area_km2"] = district_counts.area / 1000000
district_counts["density_per_100km2"] = district_counts["n_landslides"] / district_counts["area_km2"] * 100

with_records = district_counts[district_counts["n_landslides"] > 0]
with_records.sort_values("density_per_100km2", ascending=False)[["district", "n_landslides", "area_km2", "density_per_100km2"]].round(3)

Dolakha (about 2,144 km²) is slightly smaller than Chitawan (about 2,241 km²), so its density is a little higher: 0.420 against 0.402 landslides per 100 km².

The order changes in other places too. Rasuwa has only 5 records but the third-highest density, because it is a small district (about 1,501 km²).

### Practice

**Q1 · one line:** Print the largest district area in `district_counts`, rounded to the nearest km².

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(district_counts["area_km2"].max()))
```

About 7,965 km² (Dolpa). The smallest, Bhaktapur, is about 123 km², so the largest district is roughly 65 times bigger than the smallest.
</details>

**Q2 · two lines:** Which district has the **lowest** density among those with records? Sort `with_records` by density and print the `district` and `density_per_100km2` of the first row.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
lowest = with_records.sort_values("density_per_100km2").iloc[0]
print(lowest["district"], round(lowest["density_per_100km2"], 3))
```

Kaski, with 1 record in about 2,088 km²: 0.048 per 100 km².
</details>

**Q3 · three lines:** Compute the density for the whole of Nepal: total records divided by total area in km², times 100. Print it rounded to 3 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
total_n = district_counts["n_landslides"].sum()
total_km2 = district_counts["area_km2"].sum()
print(round(total_n / total_km2 * 100, 3))
```

40 records in about 148,017 km² is 0.027 per 100 km². A national figure hides the clusters completely, which is why we map densities by area.
</details>

## 3. A square grid with shapely `box`

Districts have very different sizes and shapes. A **grid** of equal cells avoids that: every cell has the same area, so counts compare directly.

`box(minx, miny, maxx, maxy)` from shapely makes one rectangle. Two nested `for` loops (A3) step across the area in x and in y and make one box per position.

Build the grid in **metres** (UTM 45N). A 10 km cell is then exactly 10,000 m × 10,000 m = 100 km², so a count per cell is already a density per 100 km².

`.total_bounds` gives the extent `[minx, miny, maxx, maxy]` (B3). Cells that do not touch the province are dropped with `.intersects()` (B7).

In [ ]:
bagmati = districts_utm[districts_utm["province_code"] == "NP03"]
bagmati_shape = bagmati.union_all()                # the 13 districts as one shape
xmin, ymin, xmax, ymax = bagmati.total_bounds
cell_m = 10000                                     # 10 km cells

cells = []
for x in np.arange(xmin, xmax, cell_m):
    for y in np.arange(ymin, ymax, cell_m):
        cells.append(box(x, y, x + cell_m, y + cell_m))

grid = gpd.GeoDataFrame(geometry=cells, crs=32645)
print(len(grid), "cells cover the bounding box")

grid = grid[grid.intersects(bagmati_shape)].copy()
grid["cell_id"] = range(len(grid))
print(len(grid), "cells touch Bagmati")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
bagmati.plot(ax=ax, color="whitesmoke", edgecolor="grey")
grid.plot(ax=ax, facecolor="none", edgecolor="teal", linewidth=0.4)
ax.set_title("10 km grid over Bagmati Province (UTM 45N)")
ax.set_axis_off()
plt.show()

The bounding box needs 459 cells; 253 of them touch Bagmati. `.copy()` makes the filtered grid a table of its own, so adding the `cell_id` column causes no warning.

`cell_id` gives each cell a name, so the counts can be merged back in the next step, as `district_code` did for districts.

### Practice

**Q1 · one line:** Print the area of the first cell of `grid` in km².

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(grid.area.iloc[0] / 1000000)
```

100.0 km²: 10 km × 10 km. Every cell has the same area, which is the whole point of a grid.
</details>

**Q2 · two lines:** How many cells lie **completely** inside Bagmati? Use `.within(bagmati_shape)` and count the `True` values.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
inside = grid.within(bagmati_shape)
print(inside.sum())
```

146 of the 253 cells lie completely inside; the other 107 are cut by the province border. Step 4 counts points in all 253, and Review 2 deals with the edge cells.
</details>

**Q3 · three lines:** Make a coarser grid: 20 km cells over the same bounding box. Store them in a list with the same loops, then print how many cells there are.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
coarse = []
for x in np.arange(xmin, xmax, 20000):
    for y in np.arange(ymin, ymax, 20000):
        coarse.append(box(x, y, x + 20000, y + 20000))
print(len(coarse))
```

126 cells instead of 459, roughly a quarter: doubling the cell side divides the number of cells by about four. Cell size is a choice you must report, because it changes the map.
</details>

## 4. Counting landslides per grid cell

Counting per cell is the B9 recipe with the grid in place of the districts.

Use an **inner** join: landslides far from Bagmati find no cell and drop out. Points just outside the border could still fall in an edge cell, so check that the total equals the number of landslides inside Bagmati.

A point lying exactly on a cell edge is a classic trap. With `predicate="within"` it matches **no** cell; with `"intersects"` it matches **two**. Checking the total catches both problems.

In [ ]:
in_cells = gpd.sjoin(landslides_utm, grid, how="inner", predicate="within")
cell_counts = in_cells.groupby("cell_id").size().reset_index(name="n_landslides")

grid_counts = grid.merge(cell_counts, on="cell_id", how="left")
grid_counts["n_landslides"] = grid_counts["n_landslides"].fillna(0).astype(int)

in_bagmati = landslides_utm.within(bagmati_shape).sum()
print(grid_counts["n_landslides"].sum(), "counted;", in_bagmati, "landslides in Bagmati")
print((grid_counts["n_landslides"] > 0).sum(), "cells have records; the busiest has", grid_counts["n_landslides"].max())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
grid_counts.plot(ax=ax, column="n_landslides", cmap="Reds", edgecolor="lightgrey", linewidth=0.3,
                 legend=True, legend_kwds={"label": "Landslides per 100 km² cell", "shrink": 0.6})
bagmati.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.6)
ax.set_title("Practice landslides per 10 km cell, Bagmati (made-up data)")
ax.set_axis_off()
plt.show()

All 26 Bagmati landslides are counted, in 16 cells. The busiest cell holds 4. Every other cell shows 0, drawn in the palest colour.

Compare this with the district map in step 1. There, all of Dolakha was dark. Here you see **where inside** Dolakha the records cluster.

### Practice

**Q1 · one line:** Print how many cells have exactly 1 landslide.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print((grid_counts["n_landslides"] == 1).sum())
```

9 cells hold a single record. Most cells with records have only one or two: the clusters are small.
</details>

**Q2 · two lines:** Show the cells with 3 or more landslides, with their `cell_id` and count.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
busy = grid_counts[grid_counts["n_landslides"] >= 3]
busy[["cell_id", "n_landslides"]]
```

Two cells qualify, one with 4 and one with 3. These are the hot spots of the practice file.
</details>

**Q3 · three lines:** Count the cells with `predicate="intersects"` instead of `"within"`, and print the total count. Is it still 26?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
touching = gpd.sjoin(landslides_utm, grid, how="inner", predicate="intersects")
print(len(touching))
print(touching.groupby("cell_id").size().sum())
```

Still 26: no practice point lies exactly on a cell edge, so both predicates agree. With real data, always compare the total with the number of points.
</details>

## 5. Hexagon grids

Many analysts prefer **hexagons** to squares. A hexagon's six neighbours all share a full side and sit at the same distance from its centre. Squares have four side neighbours and four corner neighbours, further away. Hexagon maps also show fewer straight lines that the eye mistakes for patterns.

A regular hexagon with side length `r` has six corners at angles 0°, 60°, …, 300° from its centre (A5's `math.cos` and `math.sin`). Its area is 3√3⁄2 × r². With r ≈ 6,204 m, each hexagon covers about 100 km², the same as the squares.

Hexagons tile the plane in columns 1.5 × r apart. Each column is √3 × r tall per cell, and every second column shifts up by half a cell. The function below does this for you; read it line by line.

In [ ]:
def hexagon(cx, cy, r):
    """A regular hexagon with centre (cx, cy) and side length r, in metres."""
    corners = []
    for angle in range(0, 360, 60):
        corners.append((cx + r * math.cos(math.radians(angle)),
                        cy + r * math.sin(math.radians(angle))))
    return Polygon(corners)


def hex_grid(bounds, r):
    """A list of hexagons covering bounds = [xmin, ymin, xmax, ymax]."""
    xmin, ymin, xmax, ymax = bounds
    dx = 1.5 * r
    dy = math.sqrt(3) * r
    hexes = []
    for col, x in enumerate(np.arange(xmin, xmax + dx, dx)):
        shift = dy / 2 if col % 2 == 1 else 0      # every second column moves up
        for y in np.arange(ymin - dy, ymax + dy, dy):
            hexes.append(hexagon(x, y + shift, r))
    return hexes


r_m = math.sqrt(2 * 100000000 / (3 * math.sqrt(3)))   # side for a 100 km² hexagon
print(round(r_m), "m side;", round(hexagon(0, 0, r_m).area / 1000000, 1), "km² each")

In [ ]:
hexes = gpd.GeoDataFrame(geometry=hex_grid(bagmati.total_bounds, r_m), crs=32645)
hexes = hexes[hexes.intersects(bagmati_shape)].copy()
hexes["hex_id"] = range(len(hexes))

in_hex = gpd.sjoin(landslides_utm, hexes, how="inner", predicate="within")
hex_counts = hexes.merge(in_hex.groupby("hex_id").size().reset_index(name="n_landslides"),
                         on="hex_id", how="left")
hex_counts["n_landslides"] = hex_counts["n_landslides"].fillna(0).astype(int)
print(len(hexes), "hexagons;", hex_counts["n_landslides"].sum(), "landslides counted; max", hex_counts["n_landslides"].max())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
hex_counts.plot(ax=ax, column="n_landslides", cmap="Reds", edgecolor="lightgrey", linewidth=0.3,
                legend=True, legend_kwds={"label": "Landslides per 100 km² hexagon", "shrink": 0.6})
bagmati.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.6)
ax.set_title("Practice landslides per hexagon, Bagmati (made-up data)")
ax.set_axis_off()
plt.show()

Each hexagon has a side of about 6,204 m and an area of 100.0 km². All 26 landslides are counted again.

The hot spots appear in the same places as on the square grid, but the busiest cell and the exact numbers can change, because the cell edges fall in different places. That sensitivity to where you draw the edges is real; report the cell shape and size with every grid map.

### Practice

**Q1 · one line:** Use the `hexagon` function to make one hexagon with side 1,000 m at (0, 0), and print its area in m².

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(hexagon(0, 0, 1000).area)
```

About 2,598,076 m², which is 3√3⁄2 × 1,000². A function you understand is a tool you can test with simple numbers.
</details>

**Q2 · two lines:** How many hexagons have at least one landslide? And how many squares did, in step 4? Print both.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print((hex_counts["n_landslides"] > 0).sum())
print((grid_counts["n_landslides"] > 0).sum())
```

18 hexagons but 16 squares hold records. The numbers differ because the cell edges cut the clusters in different places, even though every cell covers 100 km².
</details>

**Q3 · three lines:** Build a hexagon grid with 25 km² cells over Bagmati instead. Compute the side length, build the hexagons that touch Bagmati, and print how many there are.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
r_small_m = math.sqrt(2 * 25000000 / (3 * math.sqrt(3)))
small = gpd.GeoDataFrame(geometry=hex_grid(bagmati.total_bounds, r_small_m), crs=32645)
print(small.intersects(bagmati_shape).sum())
```

924 hexagons, against 259 with 100 km² cells: a quarter of the area per cell needs roughly four times the cells. Smaller cells show more detail but more empty cells and noisier counts.
</details>

## 6. Classifying a choropleth with mapclassify

A **choropleth** colours each area by a value. With a continuous colour bar, the eye struggles to read exact values. Most maps therefore group the values into a few **classes**, each with one colour.

**mapclassify** offers the common methods. With `k=5` classes:

- **Equal interval** splits the range from min to max into equal steps.
- **Quantiles** puts the same number of areas in each class.
- **Natural breaks** puts the breaks in the largest gaps of the data, so similar values share a class.

Printing a classifier shows its classes and how many areas fall in each. Here we classify district area in km², a value every district has.

In [ ]:
import mapclassify

area_km2 = district_counts["area_km2"]
print(mapclassify.EqualInterval(area_km2, k=5))
print(mapclassify.Quantiles(area_km2, k=5))
print(mapclassify.NaturalBreaks(area_km2, k=5))

In [ ]:
nb = mapclassify.NaturalBreaks(area_km2, k=5)
print(nb.bins.round(1))        # upper limit of each class
print(nb.counts)               # number of districts in each class

The same 77 values give three very different class tables:

- Equal interval puts 43 districts in the lowest class and only 1 in each of the two highest. A few huge mountain districts stretch the range.
- Quantiles puts 15 or 16 districts in every class, whatever the gaps.
- Natural breaks gives 7, 37, 22, 9 and 2 districts. Its upper limits are about 898, 1,712, 2,565, 3,648 and 7,965 km².

`.bins` holds the upper limit of each class and `.counts` the number of districts in each.

### Practice

**Q1 · one line:** Print a quantile classification of `area_km2` with 4 classes instead of 5.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(mapclassify.Quantiles(area_km2, k=4))
```

Each class holds 19 or 20 districts. Each class is a **quartile**: a quarter of the districts.
</details>

**Q2 · two lines:** Store an equal interval classification of `area_km2` with 5 classes, then print the width of one class: the second bin minus the first.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
ei = mapclassify.EqualInterval(area_km2, k=5)
print(round(ei.bins[1] - ei.bins[0], 1))
```

About 1,568.5 km², a fifth of the range from 122.6 to 7,965.2 km². Every class of an equal interval scheme has this width.
</details>

**Q3 · three lines:** Which class does Kathmandu fall in under natural breaks? `nb.yb` holds the class number (0 for the lowest) of every district, in table order. Add it as a column `nb_class` to `district_counts` and print Kathmandu's row.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
district_counts["nb_class"] = nb.yb
kathmandu = district_counts[district_counts["district"] == "Kathmandu"]
print(kathmandu[["district", "area_km2", "nb_class"]])
```

Kathmandu (about 414 km²) is in class 0, the smallest districts, together with the other small districts such as Bhaktapur and Lalitpur.
</details>

## 7. How classification changes the story

GeoPandas calls mapclassify for you: `.plot(column=..., scheme="naturalbreaks", k=5)`. The other names are `"equalinterval"` and `"quantiles"`. With a scheme, `legend=True` draws a legend with one box per class.

Below, the same district areas are mapped three times. Nothing about the data changes, only the classes.

**Predict:** which map will show the fewest dark districts?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
schemes = ["equalinterval", "quantiles", "naturalbreaks"]
for ax, scheme in zip(axes, schemes):
    district_counts.plot(ax=ax, column="area_km2", scheme=scheme, k=5, cmap="YlGnBu",
                         edgecolor="white", linewidth=0.3, legend=True,
                         legend_kwds={"loc": "upper center", "bbox_to_anchor": (0.5, 0.05), "ncols": 3,
                                      "fontsize": 7, "title": "District area (km²)", "fmt": "{:,.0f}"})
    ax.set_title(scheme)
    ax.set_axis_off()
plt.show()

`zip(axes, schemes)` walks through the three axes and the three names together, one pair per loop.

- **Equal interval** paints 68 of the 77 districts in the two lightest colours and only one in the darkest. Its story: "a few giant districts, and everything else small".
- **Quantiles** gives a lively map with every colour used equally. Its story: "areas vary smoothly", which hides that the largest district is about 65 times the smallest.
- **Natural breaks** sits between the two and follows the gaps in the data.

No scheme is "the truth". Pick the one that fits your purpose, name it on the map, and never shop for the scheme that best supports a conclusion you already wanted.

### Practice

**Q1 · one line:** Map `area_km2` with quantiles and 3 classes only (`k=3`), with a legend. Plot setup lines do not count.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(9, 5))
district_counts.plot(ax=ax, column="area_km2", scheme="quantiles", k=3, cmap="YlGnBu", legend=True)
plt.show()
```

Three classes give a simple "small, medium, large" map with about a third of the districts in each. Fewer classes are easier to read but hide detail.
</details>

**Q2 · two lines:** For the three schemes, print how many districts fall in the **highest** class (`.counts[-1]`). Loop over the three classifier objects.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for c in [mapclassify.EqualInterval(area_km2, k=5), mapclassify.Quantiles(area_km2, k=5), mapclassify.NaturalBreaks(area_km2, k=5)]:
    print(type(c).__name__, c.counts[-1])
```

Equal interval 1, quantiles 16, natural breaks 2. The darkest colour covers 1, 16 or 2 districts depending only on the scheme: that is how much the story can change.
</details>

**Q3 · three lines:** Classify the landslide **density** of the 8 districts with records (`with_records` from step 2) with natural breaks and `k=3`. Print the classes.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dens = with_records["density_per_100km2"]
print(mapclassify.NaturalBreaks(dens, k=3))
```

The classes hold 4, 1 and 3 districts: Kaski, Gorkha, Taplejung and Sindhupalchok up to 0.12; Myagdi alone up to 0.22; Rasuwa, Chitawan and Dolakha up to 0.42. With only 8 values, every break is fragile: one record more or less can move a district.
</details>

## 8. Honest legends

A legend is a promise about what the colours mean. An honest legend:

- states the **quantity and unit** ("landslides per 100 km²", not "density"),
- names the **classification method** or uses round, chosen breaks,
- shows **zero** as its own class, so "no records" is not mixed with "few records",
- says when the data are **made up** or incomplete.

With `scheme="userdefined"` you choose the breaks yourself through `classification_kwds={"bins": [...]}`. Each number is the upper limit of a class. `legend_kwds={"labels": [...]}` replaces the default labels with clear text.

In [ ]:
bins = [0, 0.1, 0.2, 0.3, 0.45]
labels = ["0 (no records)", "above 0 to 0.10", "0.10 to 0.20", "0.20 to 0.30", "0.30 to 0.45"]

fig, ax = plt.subplots(figsize=(10, 6))
district_counts.plot(ax=ax, column="density_per_100km2", scheme="userdefined",
                     classification_kwds={"bins": bins}, cmap="Reds",
                     edgecolor="grey", linewidth=0.3, legend=True,
                     legend_kwds={"title": "Landslides per 100 km²", "labels": labels, "loc": "lower left"})
ax.set_title("Practice landslide density by district (made-up data, 40 records)")
ax.set_axis_off()
plt.show()

In [ ]:
classes = mapclassify.UserDefined(district_counts["density_per_100km2"], bins=bins)
print(classes.counts)

Every class has a clear label with its unit. Zero has its own palest class, holding 69 districts; the other classes hold 1, 3, 1 and 3 districts.

The title says the data are made up and how many records there are. A reader who sees only this picture cannot misread it as an official landslide inventory.

### Practice

**Q1 · one line:** Print the upper limits of the classes in `classes`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(classes.bins)
```

`[0.   0.1  0.2  0.3  0.45]`: exactly the bins you chose. User-defined breaks are clear to read and stay the same if you add more data later.
</details>

**Q2 · two lines:** Make the same density map, but with natural breaks and `k=4` instead of user-defined bins. Look at the legend: what happened to the 69 zero districts?

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 6))
district_counts.plot(ax=ax, column="density_per_100km2", scheme="naturalbreaks", k=4, cmap="Reds", edgecolor="grey", linewidth=0.3, legend=True, legend_kwds={"title": "Landslides per 100 km²", "fmt": "{:.2f}"})
plt.show()
```

The zeros share the lowest class with small non-zero values, such as Kaski's 0.05, so "no records" and "some records" get the same colour. A separate zero class avoids that. `fmt` sets the number of decimals in the labels.
</details>

**Q3 · three lines:** Write the legend labels for 10 km grid cells with the bins `[0, 1, 2, 4]` (counts per 100 km² cell), then print how many cells fall in each class using `grid_counts` from step 4.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
cell_labels = ["0", "1", "2", "3 to 4"]
cell_classes = mapclassify.UserDefined(grid_counts["n_landslides"], bins=[0, 1, 2, 4])
print(cell_labels, cell_classes.counts)
```

237 empty cells, 9 cells with 1 record, 5 with 2 and 2 with 3 or 4. For whole-number counts, labels like "3 to 4" are clearer than "2.00 to 4.00".
</details>

## 9. When it breaks: a scheme name with a typo

Scheme names must be spelled exactly. Here `quantile` is missing its final `s`. Run the cell.

In [ ]:
district_counts.plot(column="area_km2", scheme="quantile", k=5)

The last line reads `ValueError: Invalid scheme: 'quantile'`, followed by a list: `Scheme must be in the set: dict_keys(['boxplot', 'equalinterval', ...])`.

A `ValueError` means the type was right (a string) but the value was not accepted. This message is unusually helpful: it lists every allowed name. Find the one you meant, `quantiles`, and copy it.

Error messages that list the allowed values are common in GeoPandas and pandas. Read them to the end.

**Try it:** fix the cell so the map draws.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines, not counting plot setup lines. Plan in words first, then write the code.

### Review 1 · A 20 km grid over Gandaki

Build a 20 km square grid over Gandaki Province (`province_code` `"NP04"`) and keep the cells that touch it. Join the landslides to the cells and compare the number counted with the number of landslides inside Gandaki. If they differ, find out why and fix it.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
gandaki = districts_utm[districts_utm["province_code"] == "NP04"]
gandaki_shape = gandaki.union_all()
gx0, gy0, gx1, gy1 = gandaki.total_bounds

g_cells = []
for x in np.arange(gx0, gx1, 20000):
    for y in np.arange(gy0, gy1, 20000):
        g_cells.append(box(x, y, x + 20000, y + 20000))
g_grid = gpd.GeoDataFrame(geometry=g_cells, crs=32645)
g_grid = g_grid[g_grid.intersects(gandaki_shape)].copy()

g_in = gpd.sjoin(landslides_utm, g_grid, how="inner", predicate="within")
print(len(g_grid), "cells;", len(g_in), "counted;", landslides_utm.within(gandaki_shape).sum(), "in Gandaki")

g_points = landslides_utm[landslides_utm.within(gandaki_shape)]          # Gandaki points only
print(len(gpd.sjoin(g_points, g_grid, how="inner", predicate="within")), "counted after selecting")
```

The 81 cells catch 17 landslides, but only 10 lie in Gandaki. Large edge cells reach into Bagmati and pick up 7 Chitawan records. Selecting the Gandaki points first gives the right 10. The total check found the problem; a grid map alone would have hidden it.
</details>

### Review 2 · Edge cells and the small-area trap

Cells on the border of Bagmati are only partly inside the province. Clip `grid_counts` to `bagmati_shape` with `gpd.clip` (B7), compute each clipped cell's area in km² and its density per 100 km², and print the 3 highest densities with their areas.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
clipped = gpd.clip(grid_counts, bagmati_shape)
clipped["area_km2"] = clipped.area / 1000000
clipped["density_per_100km2"] = clipped["n_landslides"] / clipped["area_km2"] * 100

top = clipped.sort_values("density_per_100km2", ascending=False)
print(top[["n_landslides", "area_km2", "density_per_100km2"]].head(3).round(2))
```

Clipping keeps the counts but shrinks the border cells. The top cell holds 1 record in only about 18.3 km² of Bagmati, so its density, about 5.47 per 100 km², beats the full cells with 4 and 3 records. Small areas give unstable densities; many analysts drop cells under a size limit or keep full cells.
</details>

### Review 3 · Density per province

Read `nepal_provinces.geojson` and project it to UTM 45N. Count the landslides per province with `sjoin`, merge the counts onto all 7 provinces with zeros filled in, and print each province's code, count and density per 100 km², rounded to 3 decimals.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
provinces_utm = gpd.read_file(DATA / "nepal_provinces.geojson").to_crs(32645)
p_join = gpd.sjoin(landslides_utm, provinces_utm, how="inner", predicate="within")
p_counts = p_join.groupby("province_code").size().reset_index(name="n_landslides")

provinces_utm = provinces_utm.merge(p_counts, on="province_code", how="left")
provinces_utm["n_landslides"] = provinces_utm["n_landslides"].fillna(0).astype(int)
provinces_utm["density_per_100km2"] = provinces_utm["n_landslides"] / (provinces_utm.area / 1000000) * 100
print(provinces_utm[["province_code", "n_landslides", "density_per_100km2"]].round(3))
```

Bagmati has the highest density, then Gandaki and Koshi; four provinces have 0. Larger areas smooth out clusters: the province map hides the hot spots the grid revealed.
</details>

### Review 4 · Three schemes, one table

For district area (`area_km2`), build a small table with one column per scheme (equal interval, quantiles, natural breaks, all `k=5`) holding each district's class number (`.yb`). Add the district names. Then count how many districts get the **same** class under all three schemes.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
compare = pd.DataFrame({
    "district": district_counts["district"],
    "equal": mapclassify.EqualInterval(area_km2, k=5).yb,
    "quantiles": mapclassify.Quantiles(area_km2, k=5).yb,
    "natural": mapclassify.NaturalBreaks(area_km2, k=5).yb,
})
same = (compare["equal"] == compare["quantiles"]) & (compare["quantiles"] == compare["natural"])
print(same.sum(), "districts get the same class under all three schemes")
print(compare[same].head())
```

Only 8 of the 77 districts keep the same class under all three schemes, among them Bhaktapur, Dhankuta and Dolpa. The colour a district gets depends as much on the scheme as on the district.
</details>

## Recap

- Aggregate points to areas because dots hide each other: count with `sjoin` → `groupby().size()` → `merge(how="left")` → `fillna(0)`, and check the total.
- **Density** = count ÷ area. Measure areas in UTM 45N and report a clear unit such as landslides per 100 km².
- Square grids come from `box` in nested loops; hexagons from a small function. Build both in metres; a 10 km cell is 100 km². Report the cell shape and size.
- mapclassify classifies values: equal interval, quantiles, natural breaks or your own bins. `scheme=` and `k=` in `.plot` use it directly.
- The scheme changes the story. Name it, give zero its own class, label units, and say when data are made up.

### Words you learned

| Word | Meaning |
| --- | --- |
| aggregate | Combine many records into one summary per area, such as a count |
| density | A count divided by an area, such as landslides per 100 km² |
| grid | A layer of equal cells (squares or hexagons) covering an area |
| `box` | Shapely function that makes a rectangle from minx, miny, maxx, maxy |
| hexagon | A six-sided cell; hexagons tile a surface with equal-distance neighbours |
| choropleth | A map that colours each area by a value |
| class | A range of values that shares one colour on a map |
| mapclassify | The Python library that computes map classes |
| equal interval | Classes of equal width from the minimum to the maximum |
| quantiles | Classes holding equal numbers of areas |
| natural breaks | Classes whose breaks fall in the largest gaps of the data |
| user-defined breaks | Class limits you choose yourself, usually round numbers |
| `.bins` / `.counts` / `.yb` | A classifier's class upper limits, areas per class, and class number of each area |
| `zip` | Walks through two lists together, one pair at a time |

## Next

**B11 · Making a beautiful map:** put everything together in one finished map, with layers in the right order, labels, a scale bar, a north arrow, a legend, an inset locator map and a 300 dpi export.

<!-- pygile-nav -->
**PyGILE Course** · lesson 20 of 21 · [Course home](../README.md)  
← Previous: [B9 · Spatial joins and overlays: landslides, districts and headquarters](B09_Spatial_joins_and_overlays.ipynb) · Next: [B11 · Making a beautiful map: a finished landslide map of Nepal](B11_Making_a_beautiful_map.ipynb) →